In [1]:
import requests
import re
import pandas as pd
import json

def get_uniprot(accession):
    '''define http_function to get the data from Uniprot API'''
    resp = requests.get("https://rest.uniprot.org/uniprotkb/accessions",
                        params={'accessions': accession})
    return resp

def uniprot_parse_response(resp: dict):
    '''parse response from Uniprot and output
    organism, geneInfo, sequenceInfo, type
    do not forget to include error handling (e.g. for key errors)'''
    if resp is None or resp.status_code != 200:
        return {'error': 'Invalid response'}

    try:
        data = resp.json()
    except ValueError:
        return {'error': 'Invalid JSON'}

    if 'messages' in data:
        return {'error': data['messages'][0] if data['messages'] else 'Unknown error'}
    if not data.get('results'):
        return {'error': 'No results'}

    result = data['results'][0]
    acc = result.get('primaryAccession', 'unknown')

    return {acc: {
        'organism': result.get('organism', {}).get('scientificName', 'unknown'),
        'geneInfo': result.get('genes', []),
        'sequenceInfo': result.get('sequence', {}),
        'type': result.get('entryType', 'unknown')
    }}

def get_ensembl(id):
    '''define http_function to get the data from Ensembl API'''
    clean_id = id.split('.')[0]
    resp = requests.get(f"https://rest.ensembl.org/lookup/id/{clean_id}",
                        headers={'Content-Type': 'application/json'})
    return resp

def ensembl_parse_response(resp:dict):
    '''parse Ensembl response and output
    object_type, assembly_name, species, db_type, biotype, display_name,
    id, description, canonical_transcript, source
    do not forget to include error handling (e.g. for key errors)'''
    output = {}

    if resp is None:
        return {'error': 'No response received'}

    if resp.status_code != 200:
        return {'error': f'HTTP error {resp.status_code}'}

    try:
        data = resp.json()
    except ValueError as e:
        return {'error': f'Invalid JSON response: {str(e)}'}

    if 'error' in data:
        return {'error': data['error']}

    if not data:
        return {'error': 'Empty response'}

    try:
        record_id = data.get('id', 'unknown')

        output[record_id] = {
            'object_type': data.get('object_type', 'unknown'),
            'assembly_name': data.get('assembly_name', 'unknown'),
            'species': data.get('species', 'unknown'),
            'db_type': data.get('db_type', 'unknown'),
            'biotype': data.get('biotype', 'unknown'),
            'display_name': data.get('display_name', 'unknown'),
            'id': record_id,
            'description': data.get('description', 'unknown'),
            'canonical_transcript': data.get('canonical_transcript', 'unknown'),
            'source': data.get('source', 'unknown')
        }

    except (KeyError, AttributeError, TypeError) as e:
        return {'error': f'Parsing error: {str(e)}'}

    return output

def identify_database(id_string):
    """Identify type of database by ID of record"""
    if re.match(r'^[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$', id_string, re.I):
        return 'uniprot'
    if re.match(r'^ENS[GTP]\d{11}(\.\d+)?$', id_string, re.I):
        return 'ensembl'
    return 'unknown'

def main(ids: list):
    '''
    Function that iterates over all the provided IDs and parses them into dict,
    transforms into pandas.DataFrame, and return it
    {ID : info from parse_response(), ...}

    If ID is incorrect, it should return {ID : error message}
    '''
    output = {}

    for id_str in ids:
        db_type = identify_database(id_str)

        if db_type == 'uniprot':
            response = get_uniprot(id_str)
            parsed = uniprot_parse_response(response)
            output.update(parsed)

        elif db_type == 'ensembl':
            response = get_ensembl(id_str)
            parsed = ensembl_parse_response(response)
            output.update(parsed)

        else:
            output[id_str] = {'error': 'unknown database'}

    return pd.DataFrame.from_dict(output, orient='index')

In [2]:
get_uniprot('P11473')

<Response [200]>

In [3]:
get_uniprot('helloworld')

<Response [400]>

In [4]:
get_uniprot('helloworld').json()

{'url': 'http://rest.uniprot.org/uniprotkb/accessions',
 'messages': ["Accession 'helloworld' has invalid format. It should be a valid UniProtKB accession with optional sequence range e.g. P12345[10-20]."]}

In [5]:
uniprot_parse_response(get_uniprot('P11473'))

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'UniProtKB reviewed (Swiss-Prot)'}}

In [6]:
get_ensembl('ENSMUSG00000041147')

<Response [200]>

In [7]:
get_ensembl('helloworld')

<Response [400]>

In [8]:
get_ensembl('helloworld').json()

{'error': "ID 'helloworld' not found"}

In [9]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))

{'ENSMUSG00000041147': {'object_type': 'Gene',
  'assembly_name': 'GRCm39',
  'species': 'mus_musculus',
  'db_type': 'core',
  'biotype': 'protein_coding',
  'display_name': 'Brca2',
  'id': 'ENSMUSG00000041147',
  'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
  'canonical_transcript': 'ENSMUST00000044620.11',
  'source': 'ensembl_havana'}}

In [10]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

,organism,geneInfo,sequenceInfo,type,error,object_type,assembly_name,species,db_type,biotype,display_name,id,description,canonical_transcript,source
P11473,Homo sapiens,[{'geneName': {'evidences': [{'evidenceCode': ...,{'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...,UniProtKB reviewed (Swiss-Prot),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q91XI3,Ictidomys tridecemlineatus,[{'geneName': {'value': 'INS'}}],{'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...,UniProtKB reviewed (Swiss-Prot),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hello,NaN,NaN,NaN,NaN,unknown database,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSG00000157764,NaN,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRAF,ENSG00000157764,"B-Raf proto-oncogene, serine/threonine kinase ...",ENST00000646891.2,ensembl_havana
ENSG00000139618,NaN,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRCA2,ENSG00000139618,BRCA2 DNA repair associated [Source:HGNC Symbo...,ENST00000380152.8,ensembl_havana
